In [1]:
import os
import time
from dotenv import load_dotenv
from uuid  import uuid4
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

In [3]:
load_dotenv()
pinecone_api = os.getenv("pinecone_api_key")
embeddings_hf = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
dimension=len(embeddings_hf.embed_query("test"))
print(f"Embedding dimension: {dimension}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimension: 384


In [5]:
#pinecone client
pc=Pinecone(api_key=pinecone_api) 
index_name="langchain-llama-index-v2"

In [6]:
#create a index
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    ),
    

In [ ]:
#wait for index readiness
while True:
    index_status=pc.describe_index(index_name)
    if index_status.status=="Ready":
        break
    time.sleep(2)

In [ ]:
#connect to the index
index=pc.index(index_name)

In [ ]:
vector_store = PineconeVectorStore(
    index=index,
    embedding_function=embeddings_hf,
    namespace="demo_documents",
)